In [4]:
import os
import sys
import psutil
import pyarrow as pa
from pathlib import Path
import tempfile
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
from joblib import Parallel, delayed
from multiprocessing import shared_memory
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
sys.path.insert(0, "..")
from paths import resolve

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
_NCPU = os.cpu_count() - 1 or 1
_TOTAL_MEMORY = psutil.virtual_memory().total
_AVAILABLE_MEMORY = psutil.virtual_memory().available
_MEMORY_PER_WORKER = max(100 * 1024**2, _AVAILABLE_MEMORY // (_NCPU + 1))
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | {_TOTAL_MEMORY / 1024**3:.1f}GB total RAM ({_AVAILABLE_MEMORY / 1024**3:.1f}GB available) | pyarrow {pa.__version__}", flush=True)

Running with 47 CPU cores | 92.3GB total RAM (88.2GB available) | pyarrow 24.0.0


Variables

In [5]:
%run 0_variables.ipynb

Feature ranking

In [6]:
import time as _time
_t0 = _time.perf_counter()
print(f"Loading features parquet: {os.environ['FEATURES_PATH']}", flush=True)
features = pd.read_parquet(os.environ["FEATURES_PATH"], filters=[
    ('SETTLEMENTDATE', '>=', pd.Timestamp(os.environ["FEATURE_DATASET_START"])),
    ('SETTLEMENTDATE', '<=', pd.Timestamp(os.environ["FEATIRE_DATASET_END"])),
])
features = features.drop(columns=[c for c in features.columns if c in set(os.environ["TARGET_COLS"].split(","))])
features = features.loc[:pd.Timestamp(os.environ["FEATIRE_DATASET_END"])]
print(f"  loaded features: shape={features.shape} in {_time.perf_counter() - _t0:.1f}s", flush=True)

_stem = Path(os.environ['FEATURE_DATASET']).stem
_t0 = _time.perf_counter()
print(f"Loading aggregated targets parquet for target={os.environ['TARGET']}, dataset={_stem}", flush=True)
future_prediction_targets_agg = pd.read_parquet(
    resolve(f"3_Features_select/Selected_features/{os.environ['TARGET']}_targets_agg_{_stem}.parquet")
)
print(f"  loaded targets: shape={future_prediction_targets_agg.shape} in {_time.perf_counter() - _t0:.1f}s", flush=True)

"""
Rank features

MI ranking: measures mutual information — a non-linear, information-theoretic score. It answers "does knowing this feature reduce uncertainty about the target?",
capturing non-linear dependencies too. Targets are passed in pre-aggregated to output_resolution_minutes,
so MI is computed directly at the output resolution.
"""

# Worker-side cache: each loky worker reuses its SharedMemory attachment + ndarray view
# across the many tasks it processes (avoids re-attaching/re-wrapping per task).
_WORKER_SHM_CACHE = {}

def _attach_shared(name, shape, dtype):
    cached = _WORKER_SHM_CACHE.get(name)
    if cached is not None:
        return cached[1]
    shm = shared_memory.SharedMemory(name=name)
    arr = np.ndarray(shape, dtype=dtype, buffer=shm.buf)
    _WORKER_SHM_CACHE[name] = (shm, arr)  # keep shm alive for the worker's lifetime
    return arr

def rank_features(features:pd.DataFrame,
    future_prediction_targets_agg: pd.DataFrame,
    feature_selection_subsample_start: pd.Timestamp,
    feature_selection_subsample_end: pd.Timestamp,
    feature_selection_subsample_amount: int,
    output_resolution_minutes: int,
):
    feature_cols = list(features.columns)
    target_cols_agg = list(future_prediction_targets_agg.columns)
    n_buckets = len(target_cols_agg)

    def _filter_data_for_feature_time_range_subset():
        print(f"  filtering to subsample window [{feature_selection_subsample_start} .. {feature_selection_subsample_end}]", flush=True)
        features_subset = features.loc[feature_selection_subsample_start:feature_selection_subsample_end]
        targets_subset = future_prediction_targets_agg.loc[feature_selection_subsample_start:feature_selection_subsample_end]
        shared_index = features_subset.index.intersection(targets_subset.index)
        print(f"    window rows: features={len(features_subset):,} targets={len(targets_subset):,} shared={len(shared_index):,}", flush=True)
        return features_subset.loc[shared_index], targets_subset.loc[shared_index]

    features_subset, targets_subset = _filter_data_for_feature_time_range_subset()

    # Detect discrete-looking columns BEFORE the float cast, so dtype info is preserved.
    # Integer dtypes are treated as discrete; small-cardinality float columns (<=32 uniques,
    # all integral values) are also flagged. Discrete-feature MI uses a much faster code path
    # in sklearn (no per-feature kNN noise injection / joint-space search).
    print(f"  detecting discrete features", flush=True)
    _t = _time.perf_counter()
    _discrete_mask = np.zeros(features_subset.shape[1], dtype=bool)
    for _i, _col in enumerate(feature_cols):
        _s = features_subset[_col]
        if _s.dtype.kind in ("i", "u", "b"):
            _discrete_mask[_i] = True
            continue
        if _s.dtype.kind == "f":
            _vals = _s.to_numpy()
            # cheap check: small unique count AND all integral
            if _vals.size and np.all(np.isfinite(_vals)):
                if np.unique(_vals).size <= 32 and np.all(_vals == np.floor(_vals)):
                    _discrete_mask[_i] = True
    print(
        f"    discrete features: {_discrete_mask.sum()} / {len(feature_cols)} "
        f"in {_time.perf_counter() - _t:.1f}s",
        flush=True,
    )

    # Avoid an unnecessary float32 copy when the underlying block is already float32 contiguous.
    print(f"  preparing feature ndarray (shape={features_subset.shape})", flush=True)
    _t = _time.perf_counter()
    _arr = features_subset.to_numpy(dtype=np.float32, copy=False)
    if not _arr.flags["C_CONTIGUOUS"]:
        _arr = np.ascontiguousarray(_arr)
    features_subset = _arr
    targets_subset = targets_subset.reset_index(drop=True)
    print(
        f"    ready in {_time.perf_counter() - _t:.1f}s | feature matrix "
        f"{features_subset.nbytes / 1024**2:.0f}MB | contiguous={features_subset.flags['C_CONTIGUOUS']}",
        flush=True,
    )

    def _subsample_features():
        seed = np.random.default_rng(42)
        n_samples = min(feature_selection_subsample_amount, len(features_subset))
        print(f"  subsampling {n_samples:,} of {len(features_subset):,} rows", flush=True)
        subsample_index = seed.choice(len(features_subset), size=n_samples, replace=False)
        subsample_index.sort()
        return features_subset[subsample_index], targets_subset.iloc[subsample_index]

    X_subsample, y_subsample = _subsample_features()

    def _mutual_information_scoring():
        aggregated_target_matrix = y_subsample[target_cols_agg].values.astype(np.float32)
        n_features = X_subsample.shape[1]
        n_subsample_rows = X_subsample.shape[0]

        # Chunk features so total tasks = n_buckets * n_chunks >> _NCPU. This saturates
        # all cores even when n_buckets < _NCPU and lets the scheduler load-balance
        # cheap discrete columns against expensive continuous ones.
        _target_tasks = max(_NCPU * 4, n_buckets * 2)
        _n_chunks = max(1, min(n_features, _target_tasks // max(1, n_buckets)))
        _chunk_size = max(1, (n_features + _n_chunks - 1) // _n_chunks)
        _chunks = [
            (start, min(start + _chunk_size, n_features))
            for start in range(0, n_features, _chunk_size)
        ]
        _n_tasks = len(_chunks) * n_buckets

        print(
            f"  MI scoring: {n_features} features × {n_buckets} horizons ({output_resolution_minutes}-min) "
            f"| subsample n={n_subsample_rows:,} | {_n_tasks} tasks ({len(_chunks)} feature chunks × {n_buckets} buckets) "
            f"across {_NCPU} CPUs | ~{_MEMORY_PER_WORKER / 1024**2:.0f}MB per worker",
            flush=True,
        )

        shm_X = None
        shm_y = None
        try:
            # Place arrays in shared memory so workers receive only a name string, not a pickled copy
            print(f"  allocating shared memory: X={X_subsample.nbytes / 1024**2:.0f}MB, y={aggregated_target_matrix.nbytes / 1024**2:.0f}MB", flush=True)
            _t = _time.perf_counter()
            shm_X = shared_memory.SharedMemory(create=True, size=X_subsample.nbytes)
            shm_y = shared_memory.SharedMemory(create=True, size=aggregated_target_matrix.nbytes)
            np.copyto(np.ndarray(X_subsample.shape, dtype=np.float32, buffer=shm_X.buf), X_subsample)
            np.copyto(np.ndarray(aggregated_target_matrix.shape, dtype=np.float32, buffer=shm_y.buf), aggregated_target_matrix)
            X_shape, y_shape = X_subsample.shape, aggregated_target_matrix.shape
            shm_X_name, shm_y_name = shm_X.name, shm_y.name
            print(f"    shared memory ready in {_time.perf_counter() - _t:.1f}s", flush=True)

            def _score_chunk(bucket_idx, col_start, col_end, chunk_discrete_mask):
                # Reuse worker-cached attachments across tasks (see _attach_shared).
                X = _attach_shared(shm_X_name, X_shape, np.float32)
                y = _attach_shared(shm_y_name, y_shape, np.float32)
                X_chunk = X[:, col_start:col_end]
                # Pass discrete_features as a boolean mask so sklearn uses the fast
                # discrete branch where applicable.
                result = mutual_info_regression(
                    X_chunk,
                    y[:, bucket_idx],
                    discrete_features=chunk_discrete_mask,
                    n_neighbors=3,
                    random_state=42,
                )
                return bucket_idx, col_start, col_end, result

            print(
                f"  spawning {_NCPU} loky workers and dispatching {_n_tasks} tasks "
                f"(first results may take ~30s while workers warm up)",
                flush=True,
            )
            _t_dispatch = _time.perf_counter()
            gen = Parallel(n_jobs=_NCPU, backend="loky", batch_size=1, return_as="generator_unordered", verbose=10)(
                delayed(_score_chunk)(bucket_idx, cs, ce, _discrete_mask[cs:ce])
                for bucket_idx in range(n_buckets)
                for (cs, ce) in _chunks
            )
            scores = np.empty((n_features, n_buckets))
            _first = True
            for bucket_idx, cs, ce, chunk_scores in tqdm(gen, total=_n_tasks, desc="MI scoring", leave=True):
                if _first:
                    print(f"    first worker result received after {_time.perf_counter() - _t_dispatch:.1f}s", flush=True)
                    _first = False
                scores[cs:ce, bucket_idx] = chunk_scores
        finally:
            if shm_X is not None:
                shm_X.close(); shm_X.unlink()
            if shm_y is not None:
                shm_y.close(); shm_y.unlink()

        mi_matrix = pd.DataFrame(scores, index=feature_cols, columns=target_cols_agg)
        feature_cols_ranked = mi_matrix.mean(axis=1).sort_values(ascending=False)
        return feature_cols_ranked, mi_matrix

    feature_cols_ranked, mi_matrix = _mutual_information_scoring()

    df = pd.DataFrame({
        "rank": range(1, len(feature_cols_ranked) + 1),
        "feature": feature_cols_ranked.index,
        "mean_mi": feature_cols_ranked.values,
        "target": os.environ["TARGET"],
        "feature_dataset": Path(os.environ["FEATURE_DATASET"]).stem,
    }).set_index("feature")

    df_horizons = mi_matrix.reindex(feature_cols_ranked.index)
    feature_data = pd.concat([df, df_horizons], axis=1).reset_index(names="feature")

    display(feature_data[:3])
    return feature_data

feature_data = rank_features(
    features=features,
    future_prediction_targets_agg=future_prediction_targets_agg,
    feature_selection_subsample_start=pd.Timestamp(os.environ["FEATURE_SELECTION_SUBSAMPLE_START"]),
    feature_selection_subsample_end=pd.Timestamp(os.environ["FEATURE_SELECTION_SUBSAMPLE_END"]),
    feature_selection_subsample_amount=int(os.environ["FEATURE_RANKING_SUBSAMPLE_AMOUNT"]),
    output_resolution_minutes=int(os.environ["OUTPUT_RESOLUTION"]),
)

_stem = Path(os.environ['FEATURE_DATASET']).stem
feature_data.to_parquet(
    resolve(f"3_Features_select/Selected_features/{os.environ['TARGET']}_feature_data_{_stem}.parquet")
)


Loading features parquet: /home/ec2-user/Forecasting/3_Features_select/../2_Features_build/Feature_data/1_dispatch_price.parquet
  loaded features: shape=(736417, 634) in 11.4s
Loading aggregated targets parquet for target=NSW, dataset=1_dispatch_price
  loaded targets: shape=(736417, 96) in 2.2s
  filtering to subsample window [2019-01-01 00:00:00 .. 2023-01-01 00:00:00]
    window rows: features=420,769 targets=420,769 shared=420,769
  detecting discrete features
    discrete features: 92 / 634 in 1.4s
  preparing feature ndarray (shape=(420769, 634))
    ready in 1.0s | feature matrix 1018MB | contiguous=True
  subsampling 200,000 of 420,769 rows
  MI scoring: 634 features × 96 horizons (30-min) | subsample n=200,000 | 192 tasks (2 feature chunks × 96 buckets) across 47 CPUs | ~1881MB per worker
  allocating shared memory: X=484MB, y=73MB
    shared memory ready in 0.4s
  spawning 47 loky workers and dispatching 192 tasks (first results may take ~30s while workers warm up)


[Parallel(n_jobs=47)]: Using backend LokyBackend with 47 concurrent workers.
MI scoring:   0%|          | 0/192 [00:00<?, ?it/s]

    first worker result received after 648.9s


MI scoring: 100%|██████████| 192/192 [54:14<00:00, 16.95s/it]


,feature,rank,mean_mi,target,feature_dataset,h1,h2,h3,h4,h5,h6,h7,h8,h9,h10,h11,h12,h13,h14,h15,h16,h17,h18,h19,h20,h21,h22,h23,h24,h25,h26,h27,h28,h29,h30,h31,h32,h33,h34,h35,h36,h37,h38,h39,h40,h41,h42,h43,h44,h45,h46,h47,h48,h49,h50,h51,h52,h53,h54,h55,h56,h57,h58,h59,h60,h61,h62,h63,h64,h65,h66,h67,h68,h69,h70,h71,h72,h73,h74,h75,h76,h77,h78,h79,h80,h81,h82,h83,h84,h85,h86,h87,h88,h89,h90,h91,h92,h93,h94,h95,h96
0,nsw_price_q90_336,1,0.874319,NSW,1_dispatch_price,0.986880,0.959904,0.948630,0.930107,0.922938,0.909032,0.902917,0.899473,0.897871,0.884971,0.881610,0.885368,0.877723,0.881135,0.875772,0.882695,0.880110,0.889731,0.895322,0.895525,0.890140,0.890472,0.888064,0.894254,0.879180,0.877357,0.878595,0.882497,0.875389,0.879332,0.877670,0.867428,0.867464,0.859581,0.871426,0.878560,0.882718,0.892255,0.898723,0.905172,0.908706,0.919528,0.921884,0.923974,0.927020,0.929488,0.921467,0.909195,0.896991,0.882041,0.880436,0.872465,0.866526,0.854665,0.853460,0.844097,0.844397,0.837281,0.826234,0.829145,0.833818,0.829838,0.833056,0.844320,0.841958,0.846147,0.845815,0.842111,0.846003,0.844897,0.848803,0.844459,0.845095,0.840998,0.832803,0.833381,0.831575,0.834933,0.827038,0.836323,0.827801,0.829473,0.826576,0.837572,0.838757,0.854281,0.853292,0.856542,0.865699,0.872394,0.877340,0.894922,0.891178,0.888652,0.883185,0.880610
1,nsw_price_rmean_2016,2,0.864860,NSW,1_dispatch_price,0.904853,0.897310,0.899262,0.890745,0.891508,0.880836,0.883513,0.876698,0.879701,0.870833,0.868610,0.871810,0.866583,0.870437,0.867051,0.861445,0.858313,0.863394,0.864880,0.865655,0.863756,0.872071,0.869120,0.866726,0.871377,0.868539,0.868689,0.872718,0.877758,0.876798,0.875866,0.868194,0.864097,0.863706,0.863106,0.865999,0.866596,0.860710,0.864400,0.870790,0.870288,0.867018,0.871273,0.872009,0.875709,0.877565,0.877358,0.878150,0.873640,0.880008,0.875202,0.870289,0.866619,0.864674,0.867800,0.869161,0.864469,0.863536,0.853470,0.859463,0.856082,0.854423,0.855383,0.854279,0.852119,0.851846,0.852148,0.854200,0.852092,0.854154,0.857368,0.852601,0.854037,0.858761,0.856765,0.858179,0.853347,0.852788,0.852587,0.852534,0.853447,0.847405,0.846972,0.842705,0.845906,0.850070,0.844602,0.844209,0.851161,0.851370,0.853450,0.860836,0.857167,0.858928,0.860561,0.869920
2,qld_price_q90_336,3,0.843456,NSW,1_dispatch_price,0.924981,0.905844,0.900455,0.884991,0.884430,0.872234,0.868988,0.859176,0.856589,0.846325,0.845727,0.845453,0.843117,0.849126,0.844269,0.846243,0.845815,0.849440,0.849101,0.855519,0.859230,0.863343,0.857620,0.860213,0.848557,0.846644,0.850982,0.852312,0.848041,0.851201,0.849261,0.842137,0.847173,0.840836,0.844702,0.843437,0.847129,0.852784,0.856297,0.862455,0.873953,0.882397,0.888162,0.890725,0.891093,0.892124,0.891012,0.880023,0.860789,0.859569,0.853766,0.843256,0.832326,0.823975,0.823011,0.816124,0.817010,0.813313,0.802261,0.807362,0.809433,0.809719,0.804412,0.813862,0.807870,0.807660,0.812938,0.812705,0.814101,0.816440,0.817054,0.819298,0.816893,0.819530,0.813279,0.814177,0.815362,0.810693,0.812311,0.809534,0.805207,0.808646,0.802668,0.809820,0.806399,0.823179,0.827401,0.827108,0.838874,0.851746,0.851516,0.863660,0.866229,0.863954,0.863662,0.860046
